In [ ]:
%cd C:/Users/Sadiya Maheen/Desktop/Sadiya/MedSync-AI-Chatbot/research

In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
# Extract Text from PDF Files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
# Split the Documents into Smaller Chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
texts_chunk = text_split(minimal_docs)
print(f"Number of Chunks: {len(texts_chunk)}")

In [ ]:
texts_chunk

In [91]:
from langchain.embeddings import OllamaEmbeddings
embedding = OllamaEmbeddings(model="all-minilm")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and Return the HuggingFace Embeddings Model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

In [92]:
embedding

OllamaEmbeddings(base_url='http://localhost:11434', model='all-minilm', embed_instruction='passage: ', query_instruction='query: ', mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None, show_progress=False, headers=None, model_kwargs=None)

In [93]:
vector = embedding.embed_query("Hello World")
vector

[0.260724812746048,
 0.41582930088043213,
 -0.038034677505493164,
 0.4028659462928772,
 -0.5274854302406311,
 -0.5679415464401245,
 0.554801344871521,
 0.07257382571697235,
 -0.25018632411956787,
 -0.04140598326921463,
 0.49581974744796753,
 -0.43733036518096924,
 0.435120552778244,
 -0.3452361524105072,
 -0.030293576419353485,
 -0.07349719107151031,
 0.21849019825458527,
 -0.3344104290008545,
 -0.6138172149658203,
 -0.28120580315589905,
 -0.12120947986841202,
 0.6315054893493652,
 0.0010704472661018372,
 0.2707068920135498,
 -0.21312779188156128,
 0.04624464362859726,
 0.05843089520931244,
 0.05506282299757004,
 0.393166184425354,
 -0.2980610728263855,
 -0.3665899336338043,
 0.3927936553955078,
 0.5445263981819153,
 0.486688494682312,
 0.12409461289644241,
 0.1862994134426117,
 0.10167744755744934,
 -0.5181391835212708,
 -0.224398672580719,
 -0.24119028449058533,
 0.20223523676395416,
 -0.3628253638744354,
 0.07472968101501465,
 -0.03771860897541046,
 -0.28267425298690796,
 0.04942645

In [94]:
print( "Vector Length:", len(vector))

Vector Length: 384


In [95]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [96]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [97]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [98]:
pc

In [99]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 384,  # Dimension of the Embeddings
        metric = "cosine",  # Cosine Similarity
        spec = ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [100]:
# Load Existing Index
from langchain_pinecone import PineconeVectorStore

# Embed Each Chunk and Upsert the Embeddings into Your Pinecone Index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [ ]:
# Add More Data to the Existing Pinecone Index
dswith = Document(
    page_content="dswithbappy is a youtube channel that provides tutorials on various topics.",
    metadata={"source": "Youtube"}
)

In [ ]:
docsearch.add_documents(documents=[dswith])

In [101]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [102]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='65e639b1-9c01-40f4-8cb1-1e1efdc04c90', metadata={'source': 'data\\Medical_Book-A-B.pdf'}, page_content='Researchers, Inc. Reproduced by permission.)\n26 GALE ENCYCLOPEDIA OF MEDICINE\nAcne'),
 Document(id='f8dd917f-be14-4a95-8d74-60eb556aec06', metadata={'source': 'data\\Medical_Book-C-F.pdf'}, page_content='Acne folliculitis. (Custom Medical Stock Photo. Reproduced by\npermission.)\nGALE ENCYCLOPEDIA OF MEDICINE 1503\nFood allergies'),
 Document(id='fc28e342-95bf-48aa-b54d-6650a8d20a57', metadata={'source': 'data\\Medical_Book-A-B.pdf'}, page_content='occurs when new skin cells are laid down to replace\ndamaged cells.\nThe most common sites of acne are the face, chest,\nshoulders, and back since these are the parts of the\nbody where the most sebaceous follicles are found.\nCauses and symptoms\nThe exact cause of acne is unknown. Several risk\nfactors have been identified:\n/C15Age. Due to the hormonal changes they experience,\nteenagers are more likely to develop acne.\

In [ ]:
%pip install langchain langchain-google-genai python-dotenv

In [103]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY Not Found. Please Set it in Your .env File.")

chatModel = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GEMINI_API_KEY)

system_prompt = (
    "You are a Medical Assistant for Question-Answering Tasks. "
    "Use the Following Pieces of Retrieved Context to Answer "
    "the Question. If You Don't Know the Answer, Say That You "
    "Don't Know. Use Three Sentences Maximum and Keep the "
    "Answer Concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(chatModel, prompt)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "What is Acromegaly and Gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of a chemical from the pituitary gland in the brain. This leads to increased growth in bone and soft tissue, along with other disturbances throughout the body. The same chemical is responsible for gigantism, with both conditions being linked to the overproduction of the pituitary gland.


In [104]:
response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])

Acne is a condition that commonly appears on the face, chest, shoulders, and back. These are the parts of the body where the most sebaceous follicles are found. The exact cause of acne is unknown, but hormonal changes in teenagers and gender are identified risk factors.


In [105]:
response = rag_chain.invoke({"input": "What is the Treatment of Acne?"})
print(response["answer"])

Acne treatment aims to reduce sebum production, remove dead skin cells, and kill bacteria with topical drugs and oral medications. The treatment choice depends on whether the acne is mild, moderate, or severe. Alternative treatments include proper cleansing, a well-balanced diet, and avoiding certain foods and substances.


In [106]:
response = rag_chain.invoke({"input": "What dswithbappy?"})
print(response["answer"])

I apologize, but I cannot answer your question as the term "dswithbappy" is not mentioned or defined in the provided context.


In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = Ollama(model = "mistral")

contextualize_q_system_prompt = """Given a Chat History and the Latest User Question 
Which Might Reference Context in the Chat History, Formulate a Standalone Question 
Which can be Understood without the Chat History. Do NOT Answer the Question, 
just Reformulate it if Needed and Otherwise Return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

contextualize_q_chain = contextualize_q_prompt | llm | StrOutputParser()

qa_system_prompt = """You are a Medical Assistant for Question-Answering Tasks. 
Use the Following Pieces of Retrieved Context to Answer the Question. 
If you Don't Know the Answer, Just Say that you Don't Know. 
Use Three Sentences Maximum and Keep the Answer Concise.

{context}"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def contextualized_question(input: dict):
    if input.get("chat_history"):
        return contextualize_q_chain.invoke(
            {"chat_history": input["chat_history"], "question": input["question"]}
        )
    return input["question"]

rag_chain = (
    RunnablePassthrough.assign(
        context=lambda input: format_docs(retriever.get_relevant_documents(contextualized_question(input)))
    )
    | qa_prompt
    | llm
)

In [108]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = []

question1 = "My Grandmother has a Problem with her Bladder. What Might be it?"
ai_msg1 = rag_chain.invoke({"question": question1, "chat_history": chat_history})
print(ai_msg1)

chat_history.extend([HumanMessage(content=question1), AIMessage(content=ai_msg1)])

question2 = "What Might be the Solution?"
ai_msg2 = rag_chain.invoke({"question": question2, "chat_history": chat_history})
print(ai_msg2)

chat_history.extend([HumanMessage(content=question2), AIMessage(content=ai_msg2)])

 The problem your grandmother might have is urinary incontinence, which can be caused by various factors such as neurogenic bladder, overactive bladder, or aging (common among older Americans). For more specific information, she should consult a healthcare professional. Organizations like the American Foundation for Urologic Disease may provide further resources and treatment options.
 The solution could be Bladder Training, which is often recommended for managing urge incontinence. This technique involves gradually increasing the time between urination to strengthen bladder muscles. Other treatments might include behavior modification, biofeedback, or medications. Your grandmother should consult a healthcare professional for an accurate diagnosis and personalized treatment plan.
